# Day 3 · 1교시 [실습 보조] 에이전트 기반 크롤러의 아래층 — `01_crawler`

## 실습 목표

교안 1교시에서 에이전트에게 "이 페이지 긁어줘"라고 시킨다. 이 노트북은 그 **아래층**을
직접 겪는다 — 요청→파싱→정제→Item 까지 파이썬으로. 아래층을 알고 나면 교안 [실습]에서
같은 일을 자연어로 시켜도 그 결과가 블랙박스가 아니다.

| 순서 | 내용 | 교안 연결 |
|------|------|----------|
| 1 | 요청 (httpx + 예의: User-Agent·대기) | 1.2·1.3 |
| 2 | 파싱 (BeautifulSoup 셀렉터) | 1.5 |
| 3 | 정제 (중복 제거·정규화 → Item) | 1.6 · Day2 ERD |
| 4 | (선택) LLM 한 줄 요약 | Summarizer 예고 |

> ⚙️ 대상: `quotes.toscrape.com` — **크롤링 연습용 공개 사이트**(1.3). 실제 서비스를 함부로 대상 삼지 말 것.
> 의존성: `pip install httpx beautifulsoup4`. 4절 요약은 루트 `.env`의 `MLAPI_*` 필요(없으면 건너뜀).

In [1]:
import time, httpx
from bs4 import BeautifulSoup

BASE = "https://quotes.toscrape.com/"
# 예의(1.3): 누가 왜 긁는지 밝히는 User-Agent(HTTP 헤더는 ASCII), 요청 사이 대기
HEADERS = {"User-Agent": "vibe-coding-course-crawler/1.0 (educational practice)"}

r = httpx.get(BASE, headers=HEADERS, timeout=10)
print("요청 OK:", r.status_code, "|", len(r.text), "bytes")
# robots.txt 도 코드로 확인하는 습관(예의) — 있으면 규칙을, 없으면(404) 넘어간다
rb = httpx.get(BASE + "robots.txt", headers=HEADERS, timeout=10)
print("robots.txt:", rb.text.strip()[:60] if rb.status_code == 200 else f"{rb.status_code}(없음 — 이 연습 사이트는 robots 미제공)")

요청 OK: 200 | 11021 bytes
robots.txt: 404(없음 — 이 연습 사이트는 robots 미제공)


## 2. 파싱 — 셀렉터로 필드 추출

교안 1.5에서 AI가 제안하는 그 셀렉터를 직접 쓴다. 한 명언 블록의 구조:
`div.quote` 안에 `span.text`(명언), `small.author`(저자), `div.tags a.tag`(태그들).

In [2]:
def parse_quotes(html):
    """HTML → [{text, author, tags}] (교안 1.5의 셀렉터)."""
    soup = BeautifulSoup(html, "html.parser")   # lxml 없이 표준 파서
    out = []
    for q in soup.select("div.quote"):
        out.append({
            "text":   q.select_one("span.text").get_text(strip=True),
            "author": q.select_one("small.author").get_text(strip=True),
            "tags":   [t.get_text(strip=True) for t in q.select("div.tags a.tag")],
        })
    return out

first = parse_quotes(r.text)
print(f"첫 페이지 {len(first)}건")
for q in first[:3]:
    print(f"  - ({q['author']}) {q['text'][:45]}...  tags={q['tags']}")

첫 페이지 10건
  - (Albert Einstein) “The world as we have created it is a process...  tags=['change', 'deep-thoughts', 'thinking', 'world']
  - (J.K. Rowling) “It is our choices, Harry, that show what we ...  tags=['abilities', 'choices']
  - (Albert Einstein) “There are only two ways to live your life. O...  tags=['inspirational', 'life', 'live', 'miracle', 'miracles']


## 3. 여러 페이지 + 정제 — 중복 제거·Item 매핑

여러 페이지를 **예의를 지키며**(페이지 사이 대기) 수집하고, 중복을 제거해 Day2의 데이터
모델(`Item`)로 매핑한다. 여기서 `text`를 자연 키로 중복을 거른다 — Day2 ERD의
`url UNIQUE`가 최종 방어선이라면, 크롤 단계에서 미리 거르는 것(1.6).

In [3]:
def crawl(pages=3, delay=0.5):
    """여러 페이지 수집 → 정제(중복 제거) → Item 리스트."""
    seen, items = set(), []
    for p in range(1, pages + 1):
        url = BASE if p == 1 else f"{BASE}page/{p}/"
        html = httpx.get(url, headers=HEADERS, timeout=10).text
        for q in parse_quotes(html):
            key = q["text"]                      # 자연 키(중복 판정)
            if key in seen:
                continue                          # 정제: 중복 스킵
            seen.add(key)
            items.append({                        # Day2 Item 필드로 매핑
                "source": "quotes.toscrape",
                "url": url,
                "title": q["author"],             # 저자를 title 로
                "content": q["text"],
                "tags": q["tags"],
            })
        time.sleep(delay)                         # 예의: 페이지 사이 대기
    return items

items = crawl(pages=3)
print(f"정제 후 {len(items)}건 (중복 제거됨)")
print("샘플 Item:", {k: (v[:40] if isinstance(v, str) else v) for k, v in items[0].items()})

정제 후 30건 (중복 제거됨)
샘플 Item: {'source': 'quotes.toscrape', 'url': 'https://quotes.toscrape.com/', 'title': 'Albert Einstein', 'content': '“The world as we have created it is a pr', 'tags': ['change', 'deep-thoughts', 'thinking', 'world']}


## 4. 동적 페이지



In [2]:
js_html = httpx.get("https://quotes.toscrape.com/js/",
                    headers={"User-Agent": "vibe-crawler/0.1"}, timeout=15).text
js_soup = BeautifulSoup(js_html, "html.parser")
print("정적 파싱 div.quote:", len(js_soup.select("div.quote")))

정적 파싱 div.quote: 0


In [3]:
import re, json

m = re.search(r"var data = (\[.*?\]);", js_html, re.S)   # <script> 안 JSON 덩어리
data = json.loads(m.group(1))
print("script 데이터에서:", len(data), "건")
q0 = data[0]
print(q0["author"]["name"], "—", q0["text"][:30], "...")

script 데이터에서: 10 건
Albert Einstein — “The world as we have created  ...


## 5. (선택) LLM 한 줄 요약 — Summarizer 예고

수집한 명언들을 LLM으로 한 줄 요약한다 — 관통 프로젝트 Summarizer 부품의 축소판.
(MLAPI 키 없으면 건너뜀. 이 데이터를 요약할 때가 바로 **간접 인젝션**(2교시)이 열리는 지점 —
지금은 신뢰 가능한 데이터지만, 진짜 웹은 그렇지 않다.)

In [4]:
import os, pathlib
try:
    from dotenv import load_dotenv
    load_dotenv(pathlib.Path().resolve().parents[1] / ".env")   # 루트 .env
except Exception:
    pass

if os.getenv("MLAPI_API_KEY"):
    from openai import OpenAI
    client = OpenAI(base_url=os.getenv("MLAPI_BASE_URL"), api_key=os.getenv("MLAPI_API_KEY"))
    joined = "\n".join(f"- ({it['title']}) {it['content']}" for it in items[:5])
    msg = client.chat.completions.create(
        model=os.getenv("MLAPI_MODEL", "openai/gpt-5-mini"), max_completion_tokens=2000,
        messages=[{"role": "user", "content": f"다음 명언들의 공통 주제를 한국어 한 문장으로:\n{joined}"}])
    print("요약:", msg.choices[0].message.content.strip())
else:
    print("MLAPI_* 키 없음 → 4절 건너뜀 (graceful)")

요약: 이 명언들의 공통 주제는 우리의 생각과 선택, 즉 관점과 진정성이 삶의 모습·정체성·아름다움을 만들어낸다는 것이다.


## 실습 정리

- **요청·파싱·정제**를 직접 — 에이전트가 "긁어줘"로 해 주는 일의 밑바닥.
- **예의**(User-Agent·대기·robots)는 코드에 명시했다 — 책임은 사람에게(1.3).
- 정제된 Item 이 Day2 ERD의 행이 된다 — 문서(설계)→코드(크롤러)→실물(DB)의 연결.
- 이제 교안 1교시로: 같은 파서를 **에이전트에게** 시키고, `crawler/` 스텁을 테스트 통과로 채운다.
- 4절에서 이 데이터를 LLM에 넣었다 — **외부 데이터를 요약하는 순간이 2교시(보안)의 무대**다.